# 02 — Modèle RNN

**Prérequis** : `01_clustering.ipynb` exécuté.

**Données** : `../data/client_data.pkl`, `../data/vocab.pkl`, `../data/label_maps.json`

**Sortie** : `../models/rnn_best.pt`, `../data/results_rnn.pkl`

## Bloc 1 : Config & imports

In [ ]:
import os, pickle, json, math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR  = '../data/'
MODEL_DIR = '../models/'
os.makedirs(MODEL_DIR, exist_ok=True)

# Hyperparamètres
EMBED_DIM    = 128
HIDDEN_DIM   = 256
N_CLASSES    = 8
DROPOUT      = 0.3
BATCH_SIZE   = 64
LR           = 1e-3
N_EPOCHS     = 10
RANDOM_STATE = 42

torch.manual_seed(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

## Bloc 2 : Chargement des données

In [ ]:
with open(DATA_DIR + 'client_data.pkl', 'rb') as f:
    data = pickle.load(f)
with open(DATA_DIR + 'vocab.pkl', 'rb') as f:
    vocab = pickle.load(f)
with open(DATA_DIR + 'label_maps.json') as f:
    label_maps = json.load(f)

CLIENT_LABELS = {int(k): v for k, v in label_maps['client'].items()}
label_names   = [CLIENT_LABELS[i] for i in range(N_CLASSES)]
PAD_IDX = vocab['<PAD>']

print(f'Vocab : {len(vocab):,} tokens | PAD_IDX={PAD_IDX}')
print(f'Train : {len(data["train"]):,} | Val : {len(data["val"]):,} | Test : {len(data["test"]):,}')
print(f'Classes ({N_CLASSES}) : {label_names}')

## Bloc 3 — Dataset + DataLoader (pattern TP5)

In [ ]:
class IntentDataset(Dataset):
    def __init__(self, data):
        self.texts  = [d[0] for d in data]
        self.labels = [d[1] for d in data]
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx): return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts, labels = zip(*batch)
    texts_padded  = pad_sequence(
        [torch.tensor(t, dtype=torch.long) for t in texts],
        batch_first=True, padding_value=PAD_IDX
    )
    return texts_padded, torch.tensor(labels, dtype=torch.long)

loaders = {
    split: DataLoader(
        IntentDataset(data[split]),
        batch_size=BATCH_SIZE,
        shuffle=(split == 'train'),
        collate_fn=collate_fn
    )
    for split in ('train', 'val', 'test')
}

for split, loader in loaders.items():
    x, y = next(iter(loader))
    print(f'{split:5s} — textes: {x.shape}, labels: {y.shape}')

## Bloc 4 : Modèle RNN (pattern TP5)

In [ ]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, dropout=0.3):
        super(RNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn       = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded       = self.embedding(x)           # (batch, seq, embed_dim)
        output, hidden = self.rnn(embedded)          # output : (batch, seq, hidden_dim)
        # Mean pooling sur toute la séquence (plus robuste que hidden[-1] pour 8 classes)
        # hidden[-1] perd les mots du début à cause du vanishing gradient sur 64 tokens
        return self.fc(self.dropout(output.mean(dim=1)))  # (batch, output_dim)

model = RNN(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    output_dim=N_CLASSES,
    dropout=DROPOUT
).to(device)

print(model)
print(f'Paramètres totaux : {sum(p.numel() for p in model.parameters()):,}')

## Bloc 5 : Entraînement (pattern TP5)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

train_losses, val_losses, val_accs = [], [], []
best_val_acc = 0.0

for epoch in range(N_EPOCHS):
    #  Train 
    model.train()
    total_loss = 0.0
    for texts, labels in loaders['train']:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(texts), labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # RNN : évite gradient explosion
        optimizer.step()
        total_loss += loss.item()
    train_losses.append(total_loss / len(loaders['train']))

    #  Validation 
    model.eval()
    val_loss, preds, targets = 0.0, [], []
    with torch.no_grad():
        for texts, labels in loaders['val']:
            texts, labels = texts.to(device), labels.to(device)
            out     = model(texts)
            val_loss += criterion(out, labels).item()
            preds.extend(torch.argmax(out, dim=1).cpu().tolist())
            targets.extend(labels.cpu().tolist())
    val_losses.append(val_loss / len(loaders['val']))
    acc = accuracy_score(targets, preds)
    val_accs.append(acc)

    print(f'Epoch {epoch+1:2d}/{N_EPOCHS} | train={train_losses[-1]:.4f} | val={val_losses[-1]:.4f} | val_acc={acc:.4f}', end='')

    if acc > best_val_acc:
        best_val_acc = acc
        torch.save(model.state_dict(), MODEL_DIR + 'rnn_best.pt')
        print(' ✓ sauvegardé')
    else:
        print()

## Bloc 6 : Courbes d'entraînement

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses, label='Train'); axes[0].plot(val_losses, label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(val_accs, color='green')
axes[1].set_title('Accuracy validation'); axes[1].set_xlabel('Epoch')
axes[1].axhline(best_val_acc, color='red', linestyle='--', label=f'Best={best_val_acc:.4f}')
axes[1].legend()
plt.suptitle('RNN — Courbes entraînement')
plt.tight_layout()
plt.savefig(MODEL_DIR + 'rnn_curves.png', dpi=100)
plt.show()

## Bloc 7 : Évaluation sur le test set

In [ ]:
model.load_state_dict(torch.load(MODEL_DIR + 'rnn_best.pt', map_location=device))
model.eval()

preds, targets = [], []
with torch.no_grad():
    for texts, labels in loaders['test']:
        texts, labels = texts.to(device), labels.to(device)
        out = model(texts)
        preds.extend(torch.argmax(out, dim=1).cpu().tolist())
        targets.extend(labels.cpu().tolist())

test_acc = accuracy_score(targets, preds)
report   = classification_report(targets, preds, target_names=label_names)
print(f'Test Accuracy : {test_acc:.4f}\n')
print(report)

cm = confusion_matrix(targets, preds)
plt.figure(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)
plt.title(f'Matrice de confusion — RNN (test_acc={test_acc:.4f})')
plt.ylabel('Vrai'); plt.xlabel('Prédit')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(MODEL_DIR + 'rnn_confusion.png', dpi=100)
plt.show()

## Bloc 8 : Sauvegarde des résultats

In [ ]:
results_rnn = {
    'model'       : 'RNN',
    'test_accuracy': test_acc,
    'best_val_acc': best_val_acc,
    'train_losses': train_losses,
    'val_losses'  : val_losses,
    'val_accs'    : val_accs,
    'predictions' : preds,
    'targets'     : targets,
    'report'      : report,
}
with open(DATA_DIR + 'results_rnn.pkl', 'wb') as f:
    pickle.dump(results_rnn, f)

print(f'Résultats sauvegardés → {DATA_DIR}results_rnn.pkl')
print(f'Modèle sauvegardé    → {MODEL_DIR}rnn_best.pt')
print(f'Test accuracy : {test_acc:.4f}')